In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import joblib
import optuna
from optuna.pruners import MedianPruner
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, auc
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Helper Classes

In [2]:
class IsolationForestModel:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_train = None
        self.df_valid = None
        self.df_test = None
        self.list_feature_cols = None
        self.model = None
        self.arr_train_scores = None
        self.arr_valid_scores = None
        self.arr_test_scores = None
        self.flt_optimal_threshold = None
        self.dict_best_params = None

    def import_data(self):
        """Load preprocessed data from S3."""
        print("Loading preprocessed data from S3...")
        str_train_uri = f's3://{self.str_bucket}/02_preprocessing/train_processed.parquet'
        str_valid_uri = f's3://{self.str_bucket}/02_preprocessing/valid_processed.parquet'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_processed.parquet'
        
        self.df_train = pd.read_parquet(str_train_uri)
        self.df_valid = pd.read_parquet(str_valid_uri)
        self.df_test = pd.read_parquet(str_test_uri)
        
        self.list_feature_cols = [col for col in self.df_train.columns if col != 'isFraud']
        
        print(f"Training set: {self.df_train.shape}")
        print(f"Validation set: {self.df_valid.shape}")
        print(f"Test set: {self.df_test.shape}")
        print(f"Features: {len(self.list_feature_cols)}")
        
        return self.df_train, self.df_valid, self.df_test

    def tune_model(self, int_n_trials=20):
        """Tune hyperparameters using Optuna on validation set."""
        if self.df_train is None:
            raise ValueError("Data not loaded.")
        
        print(f"\nTuning Isolation Forest with {int_n_trials} trials (using validation set)...")
        
        def objective(trial):
            int_n_estimators = trial.suggest_int('n_estimators', 50, 300, step=10)
            flt_max_samples = trial.suggest_float('max_samples', 0.1, 1.0)
            flt_contamination = trial.suggest_float('contamination', 0.001, 0.1)
            int_max_features = trial.suggest_int('max_features', 1, len(self.list_feature_cols))
            
            model_temp = IsolationForest(
                n_estimators=int_n_estimators,
                max_samples=flt_max_samples,
                contamination=flt_contamination,
                max_features=int_max_features,
                random_state=42,
                n_jobs=-1
            )
            
            model_temp.fit(self.df_train[self.list_feature_cols])
            arr_scores_valid = model_temp.score_samples(self.df_valid[self.list_feature_cols])
            
            # Use PR-AUC on validation set for objective
            arr_precision, arr_recall, _ = precision_recall_curve(self.df_valid['isFraud'].values, -arr_scores_valid)
            flt_pr_auc = auc(arr_recall, arr_precision)
            return flt_pr_auc
        
        study = optuna.create_study(direction='maximize', pruner=MedianPruner())
        study.optimize(objective, n_trials=int_n_trials, show_progress_bar=True)
        
        self.dict_best_params = study.best_params
        print(f"\nBest parameters found:")
        for str_key, val in self.dict_best_params.items():
            print(f"  {str_key}: {val}")
        print(f"Best Validation PR-AUC: {study.best_value:.4f}")
        
        return self.dict_best_params

    def fit_model(self):
        """Fit the Isolation Forest model with best parameters."""
        if self.df_train is None:
            raise ValueError("Data not loaded.")
        if self.dict_best_params is None:
            raise ValueError("Model not tuned. Call tune_model() first.")
        
        print("\nFitting Isolation Forest model...")
        
        self.model = IsolationForest(
            n_estimators=self.dict_best_params['n_estimators'],
            max_samples=self.dict_best_params['max_samples'],
            contamination=self.dict_best_params['contamination'],
            max_features=self.dict_best_params['max_features'],
            random_state=42,
            n_jobs=-1
        )
        
        self.model.fit(self.df_train[self.list_feature_cols])
        print("Model fitted successfully")
        
        return self.model

    def predict(self):
        """Generate anomaly scores for train, valid, and test sets."""
        if self.model is None:
            raise ValueError("Model not fitted.")
        
        print("\nGenerating anomaly scores...")
        
        self.arr_train_scores = self.model.score_samples(self.df_train[self.list_feature_cols])
        self.arr_valid_scores = self.model.score_samples(self.df_valid[self.list_feature_cols])
        self.arr_test_scores = self.model.score_samples(self.df_test[self.list_feature_cols])
        
        print(f"Train scores - Mean: {self.arr_train_scores.mean():.4f}, Std: {self.arr_train_scores.std():.4f}")
        print(f"Valid scores - Mean: {self.arr_valid_scores.mean():.4f}, Std: {self.arr_valid_scores.std():.4f}")
        print(f"Test scores - Mean: {self.arr_test_scores.mean():.4f}, Std: {self.arr_test_scores.std():.4f}")
        
        return self.arr_train_scores, self.arr_valid_scores, self.arr_test_scores

    def evaluate(self):
        """Evaluate model on test set using threshold selected on validation set."""
        if self.arr_test_scores is None or self.arr_valid_scores is None:
            raise ValueError("Predictions not generated.")
        
        print("\nEvaluating model...")
        
        # Find optimal threshold on validation set
        arr_y_valid = self.df_valid['isFraud'].values
        arr_valid_scores_neg = -self.arr_valid_scores
        
        arr_precision_v, arr_recall_v, arr_thresholds_v = precision_recall_curve(arr_y_valid, arr_valid_scores_neg)
        
        # Compute F1 from precision/recall arrays (vectorized, no loop)
        arr_f1_v = 2 * (arr_precision_v[:-1] * arr_recall_v[:-1]) / (arr_precision_v[:-1] + arr_recall_v[:-1] + 1e-10)
        int_best_idx = np.argmax(arr_f1_v)
        self.flt_optimal_threshold = arr_thresholds_v[int_best_idx]
        print(f"Optimal Threshold (from validation set): {self.flt_optimal_threshold:.4f}")
        
        # Evaluate on test set using that threshold
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        flt_roc_auc = roc_auc_score(arr_y_test, arr_scores_neg)
        print(f"Test ROC-AUC: {flt_roc_auc:.4f}")
        
        arr_precision, arr_recall, arr_thresholds = precision_recall_curve(arr_y_test, arr_scores_neg)
        flt_pr_auc = auc(arr_recall, arr_precision)
        print(f"Test PR-AUC: {flt_pr_auc:.4f}")
        
        arr_preds_optimal = (arr_scores_neg >= self.flt_optimal_threshold).astype(int)
        
        flt_precision = precision_score(arr_y_test, arr_preds_optimal)
        flt_recall = recall_score(arr_y_test, arr_preds_optimal)
        flt_f1 = f1_score(arr_y_test, arr_preds_optimal)
        
        print(f"Test Precision: {flt_precision:.4f}")
        print(f"Test Recall: {flt_recall:.4f}")
        print(f"Test F1-Score: {flt_f1:.4f}")
        
        return {
            'roc_auc': flt_roc_auc,
            'pr_auc': flt_pr_auc,
            'precision': flt_precision,
            'recall': flt_recall,
            'f1': flt_f1,
            'threshold': self.flt_optimal_threshold
        }

    def plot_anomaly_scores(self):
        """Plot distribution of anomaly scores."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        ax.hist(-self.arr_test_scores[arr_y_test == 0], bins=100, alpha=0.6, label='Non-Fraud', color='#2ecc71')
        ax.hist(-self.arr_test_scores[arr_y_test == 1], bins=100, alpha=0.6, label='Fraud', color='#e74c3c')
        
        ax.axvline(self.flt_optimal_threshold, color='black', linestyle='--', linewidth=2, label=f'Optimal Threshold: {self.flt_optimal_threshold:.4f}')
        
        ax.set_xlabel('Anomaly Score', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax.set_title('Isolation Forest Anomaly Score Distribution', fontsize=14, fontweight='bold')
        ax.legend()
        ax.set_yscale('log')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_anomaly_scores.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 01_anomaly_scores.png")

    def plot_precision_recall_curve(self):
        """Plot precision-recall curve."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        arr_precision, arr_recall, arr_thresholds = precision_recall_curve(arr_y_test, arr_scores_neg)
        flt_pr_auc = auc(arr_recall, arr_precision)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_recall, arr_precision, linewidth=2.5, label=f'PR Curve (AUC={flt_pr_auc:.4f})', color='steelblue')
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
        
        ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
        ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
        ax.set_title('Precision-Recall Curve - Isolation Forest', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_precision_recall_curve.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 02_precision_recall_curve.png")

    def plot_roc_curve(self):
        """Plot ROC curve."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        flt_roc_auc = roc_auc_score(arr_y_test, arr_scores_neg)
        arr_fpr, arr_tpr, arr_thresholds = roc_curve(arr_y_test, arr_scores_neg)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_fpr, arr_tpr, linewidth=2.5, label=f'ROC Curve (AUC={flt_roc_auc:.4f})', color='steelblue')
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5, label='Random Classifier')
        
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curve - Isolation Forest', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_roc_curve.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 03_roc_curve.png")

    def plot_confusion_matrix(self):
        """Plot confusion matrix at optimal threshold."""
        if self.arr_test_scores is None or self.flt_optimal_threshold is None:
            raise ValueError("Predictions not generated or threshold not set.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        arr_preds = (arr_scores_neg >= self.flt_optimal_threshold).astype(int)
        
        cm = confusion_matrix(arr_y_test, arr_preds)
        
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                    xticklabels=['Non-Fraud', 'Fraud'],
                    yticklabels=['Non-Fraud', 'Fraud'])
        
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_title('Confusion Matrix - Isolation Forest', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_confusion_matrix.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 04_confusion_matrix.png")

    def save_model(self):
        """Save trained model to disk and S3."""
        if self.model is None:
            raise ValueError("Model not fitted.")
        
        print("\nSaving model...")
        
        from io import BytesIO
        buffer = BytesIO()
        joblib.dump(self.model, buffer)
        buffer.seek(0)
        
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key='03_isolation_forest/isolation_forest_model.pkl',
                Body=buffer.getvalue()
            )
            print("Model saved to S3")
        except Exception as e:
            print(f"Error uploading to S3: {e}")

## Constants

In [3]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'isolation_forest'
str_dirname_output = './output'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

Output directory ready: ./output


## Load Data

In [5]:
model_if = IsolationForestModel(str_bucket, str_dirname_output)
df_train, df_valid, df_test = model_if.import_data()

Loading preprocessed data from S3...
Training set: (600593, 11)
Validation set: (228103, 11)
Test set: (123580, 11)
Features: 10


## Hyperparameter Tuning

In [6]:
dict_best_params = model_if.tune_model(int_n_trials=5)

[I 2026-03-22 00:39:49,603] A new study created in memory with name: no-name-badd33c7-b7ba-44bc-8089-4771f56881dd



Tuning Isolation Forest with 5 trials (using validation set)...


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-03-22 00:41:27,459] Trial 0 finished with value: 0.4696748306141324 and parameters: {'n_estimators': 260, 'max_samples': 0.9378124018039901, 'contamination': 0.013335065608183296, 'max_features': 8}. Best is trial 0 with value: 0.4696748306141324.
[I 2026-03-22 00:41:47,010] Trial 1 finished with value: 0.3330988357911875 and parameters: {'n_estimators': 80, 'max_samples': 0.5176316140170487, 'contamination': 0.09622940458774837, 'max_features': 2}. Best is trial 0 with value: 0.4696748306141324.
[I 2026-03-22 00:42:26,111] Trial 2 finished with value: 0.20668925536099925 and parameters: {'n_estimators': 280, 'max_samples': 0.16400264446868734, 'contamination': 0.02965995556361767, 'max_features': 2}. Best is trial 0 with value: 0.4696748306141324.
[I 2026-03-22 00:42:53,221] Trial 3 finished with value: 0.3444755051399626 and parameters: {'n_estimators': 90, 'max_samples': 0.7677089108221203, 'contamination': 0.027700063844246295, 'max_features': 2}. Best is trial 0 with value

## Fit Model

In [7]:
model = model_if.fit_model()


Fitting Isolation Forest model...
Model fitted successfully


## Generate Predictions

In [8]:
arr_train_scores, arr_valid_scores, arr_test_scores = model_if.predict()


Generating anomaly scores...
Train scores - Mean: -0.3516, Std: 0.0313
Valid scores - Mean: -0.3535, Std: 0.0354
Test scores - Mean: -0.3537, Std: 0.0386


## Model Evaluation

In [9]:
dict_metrics = model_if.evaluate()


Evaluating model...
Optimal Threshold (from validation set): 0.5392
Test ROC-AUC: 0.9140
Test PR-AUC: 0.5182
Test Precision: 0.7912
Test Recall: 0.3918
Test F1-Score: 0.5241


## Generate Visualizations

In [10]:
model_if.plot_anomaly_scores()

Saved: 01_anomaly_scores.png


In [11]:
model_if.plot_precision_recall_curve()

Saved: 02_precision_recall_curve.png


In [12]:
model_if.plot_roc_curve()

Saved: 03_roc_curve.png


In [13]:
model_if.plot_confusion_matrix()

Saved: 04_confusion_matrix.png


## Save Model

In [14]:
model_if.save_model()


Saving model...
Model saved to S3


## Completion

In [15]:
print("\n=== ISOLATION FOREST MODELING COMPLETE ===")
print(f"\nMetrics Summary:")
for str_metric, val in dict_metrics.items():
    print(f"  {str_metric}: {val:.4f}")


=== ISOLATION FOREST MODELING COMPLETE ===

Metrics Summary:
  roc_auc: 0.9140
  pr_auc: 0.5182
  precision: 0.7912
  recall: 0.3918
  f1: 0.5241
  threshold: 0.5392
